# Data Handling Pipeline

This notebook runs one single pipeline for the selected coins.

Pipeline order:
1. Select coins and shared parameters
2. Build daily RV from 5-minute raw data
3. Detect intraday downside extreme events with rolling quantile regression
4. Aggregate intraday events into daily features
5. Compute Hawkes intensity (`lambda_t`) and align it into `event_5_min` and `daily_rv`

Run the notebook top-to-bottom. Change the coin list only in Cell 2.

In [15]:
import importlib
import sys
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import data.build_intraday_and_rv as build_intraday_and_rv_module
import features.identify_events as identify_events_module
import features.create_raw_event_feat as create_raw_event_feat_module
import features.create_hawkes_feat as create_hawkes_feat_module

# Force reload so the notebook uses the latest pipeline code and schema names.
build_intraday_and_rv_module = importlib.reload(build_intraday_and_rv_module)
identify_events_module = importlib.reload(identify_events_module)
create_raw_event_feat_module = importlib.reload(create_raw_event_feat_module)
create_hawkes_feat_module = importlib.reload(create_hawkes_feat_module)

build_intraday_and_rv = build_intraday_and_rv_module.build_intraday_and_rv
rolling_quantile_regression_window = identify_events_module.rolling_quantile_regression_window
build_daily_n_events_feature = create_raw_event_feat_module.build_daily_n_events_feature
build_daily_hawkes_feature = create_hawkes_feat_module.build_daily_hawkes_feature

# -----------------------------------------------------------------------------
# GLOBAL PIPELINE CONFIGURATION
# -----------------------------------------------------------------------------
COINS_TO_USE = ["BTC", "ETH"]
RAW_DIR = "../data/raw"
EVENT_5M_DIR = "../data/interim/event_5_min"

# Rolling quantile-regression parameters
TAU = 0.95
TRAIN_DAYS = 7
TEST_DAYS = 1
INTERVALS_PER_DAY = 288

# Hawkes parameters
HAWKES_CONDA_ENV = "hawkes_env"
HAWKES_BETA = 1.5
HAWKES_GOFIT = "least-squares"
HAWKES_GRID_MINUTES = 5
HAWKES_LOOKBACK_DAYS = 30
HAWKES_MAX_ITER = 2000
HAWKES_OVERWRITE = False

print("Pipeline configuration loaded")
print(f"Coins: {COINS_TO_USE}")
print(f"Raw dir: {RAW_DIR}")
print(f"Event dir: {EVENT_5M_DIR}")
print(
    f"QReg params -> tau={TAU}, train_days={TRAIN_DAYS}, test_days={TEST_DAYS}, intervals_per_day={INTERVALS_PER_DAY}"
)
print(
    f"Hawkes params -> env={HAWKES_CONDA_ENV}, beta={HAWKES_BETA}, lookback_days={HAWKES_LOOKBACK_DAYS}"
)

Pipeline configuration loaded
Coins: ['BTC', 'ETH']
Raw dir: ../data/raw
Event dir: ../data/interim/event_5_min
QReg params -> tau=0.95, train_days=7, test_days=1, intervals_per_day=288
Hawkes params -> env=hawkes_env, beta=1.5, lookback_days=30


## Step 1. Build Daily RV

For every coin in `COINS_TO_USE`, build the daily RV parquet from the raw 5-minute data.

In [16]:
# Step 1: Build daily RV parquet files from raw 5-minute OHLCV data.
for coin_name in COINS_TO_USE:
    print(f"Processing daily RV for {coin_name}...")
    build_intraday_and_rv(coin_name)

print("Step 1 completed: daily RV files updated.")

Processing daily RV for BTC...
Raw intraday file used: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\raw\BTCUSDT_binance_5min_2017_2026.csv
Daily RV saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\BTC_rv.parquet
Data quality summary saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\data_quality\BTC\data_quality_summary.csv
Daily coverage details saved to: C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\results\data_quality\BTC\daily_coverage.csv
Valid interval for BTC: 2017-08-17 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00
Processing daily RV for 

## Step 2. Detect Intraday Downside Events

Using the shared quantile-regression settings from Cell 2, detect downside events for each selected coin and save them to `data/interim/event_5_min/{COIN}_5m_events.parquet`.

In [17]:
# Step 2: Detect downside extreme events at 5-minute frequency.
for coin_name in COINS_TO_USE:
    results_df, predictions_df, pinball, interim_path = rolling_quantile_regression_window(
        coin=coin_name,
        tau=TAU,
        N=TRAIN_DAYS,
        P=TEST_DAYS,
        D=INTERVALS_PER_DAY,
        raw_dir=RAW_DIR,
        interim_dir=EVENT_5M_DIR,
    )

    print(f"\n{'=' * 60}")
    print(f"Step 2 results for coin={coin_name}")
    print(f"{'=' * 60}")
    print(f"Total rows: {len(results_df)}, Predicted rows: {len(predictions_df)}")
    print(f"tau={TAU} | rolling pinball loss={pinball:.8f}")
    print(f"Interim file: {interim_path}")

print("Step 2 completed: intraday event files updated.")

Loading data for BTC...
Found existing interim data with 877262 rows
Resuming from rolling window 3053
Starting rolling window: tau=0.95, N=7 days, P=1 days
Total chunks to process: 3053
Aggregating results...
Completed! Interim results saved to: ..\data\interim\event_5_min\BTC_5m_events.parquet

Step 2 results for coin=BTC
Total rows: 879227, Predicted rows: 0
tau=0.95 | rolling pinball loss=nan
Interim file: ..\data\interim\event_5_min\BTC_5m_events.parquet
Loading data for ETH...
Found existing interim data with 877262 rows
Resuming from rolling window 3053
Starting rolling window: tau=0.95, N=7 days, P=1 days
Total chunks to process: 3053
Aggregating results...
Completed! Interim results saved to: ..\data\interim\event_5_min\ETH_5m_events.parquet

Step 2 results for coin=ETH
Total rows: 879227, Predicted rows: 0
tau=0.95 | rolling pinball loss=nan
Interim file: ..\data\interim\event_5_min\ETH_5m_events.parquet
Step 2 completed: intraday event files updated.


## Step 3. Aggregate Intraday Events into Daily Features

Merge the daily downside-event features (`ratio_event`, `exceedance_down`) into the daily RV parquet for each coin in `COINS_TO_USE`.

In [18]:
# Step 3: Merge daily downside-event features into daily RV files.
for coin_name in COINS_TO_USE:
    daily_rv_with_events = build_daily_n_events_feature(coin=coin_name)
    print(
        f"Updated daily RV with downside event features for {coin_name} | rows={len(daily_rv_with_events)}"
    )

print("Step 3 completed: daily RV files now include ratio_event and exceedance_down.")

BTC: added 'ratio_event', 'exceedance_down' to C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\BTC_rv.parquet
Updated daily RV with downside event features for BTC | rows=3059
ETH: added 'ratio_event', 'exceedance_down' to C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\ETH_rv.parquet
Updated daily RV with downside event features for ETH | rows=3059
Step 3 completed: daily RV files now include ratio_event and exceedance_down.


## Step 4. Compute and Align Hawkes Feature

Using the shared Hawkes settings from Cell 2, compute `lambda_t` for each selected coin and align it into:
- `event_5_min` files
- `daily_rv` files

In [19]:
# Step 4: Compute Hawkes intensity and align it into event_5_min and daily_rv.
for coin_name in COINS_TO_USE:
    updated_df = build_daily_hawkes_feature(
        coin=coin_name,
        beta=HAWKES_BETA,
        gofit=HAWKES_GOFIT,
        grid_minutes=HAWKES_GRID_MINUTES,
        lookback_days=HAWKES_LOOKBACK_DAYS,
        max_iter=HAWKES_MAX_ITER,
        hawkes_conda_env=HAWKES_CONDA_ENV,
        overwrite=HAWKES_OVERWRITE,
        verbose=True,
    )
    print(f"Updated Hawkes feature for {coin_name}")
    print(updated_df[["lambda_t"]].tail(3))

print("Step 4 completed: lambda_t aligned into event_5_min and daily_rv.")

Saved 3023 Hawkes diagnostics rows to C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\temp\hawkes\BTC_hawkes_diagnostics.parquet.
BTC: aligned 'lambda_t' into C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\event_5_min\BTC_5m_events.parquet
BTC: added 'lambda_t' to C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) Proyectos\(03) Github\Finanzas\Hawkes Enhanced RV Forecast\data\interim\daily_rv\BTC_rv.parquet
Updated Hawkes feature for BTC
                           lambda_t
date                               
2025-12-29 00:00:00+00:00  0.720465
2025-12-30 00:00:00+00:00  0.713374
2025-12-31 00:00:00+00:00  0.722948
Saved 3023 Hawkes diagnostics rows to C:\Users\diego\OneDrive - Universidad Técnica Federico Santa María\(00)-Documents\(01) P